# HMS v6 — matched no-reweight baselines and extra training seeds (200 s, boundary-agnostic)

**Inputs:** (1) HMS competition data; (2) output of the v5 notebook (`notebookbb6726eb71`, holds `results_v5/*_valpred.npy`). To resume a run that stopped at the time budget, also attach this notebook's own output. GPU T4, **Save Version → Save & Run All**.

**Why.**
- v5 trained the no-reweight variant (the recommended recipe) only for Mamba. GRU and Transformer get it here, so the architecture comparison uses the same recipe for everyone.
- Every model so far was trained with one seed. Two extra training seeds (same folds) separate architecture effects from training noise.

**Time.** Runs are ordered by priority and the loop stops before a run that would exceed `TIME_BUDGET_H` (default 4.5 h, est. per fold: Mamba 16 min, GRU 4, Transformer 5). Every finished run is saved immediately, so nothing is lost if it stops early.

In [1]:
import os, glob, json, math, time, random, shutil
import numpy as np
import pandas as pd

CFG = dict(
    DATA_ROOT=None,                       # None = auto-detect under /kaggle/input
    CACHE_DIR="/kaggle/working/cache",    # replaced in the v2 settings cell below
    RESULTS_DIR="/kaggle/working/results",
    FS=100,                               # target sampling rate (Hz)
    CTX_SEC=50,
    CACHE_CTX_SEC=100,
    JITTER_SEC=1.0,                       # train-time random shift of the window centre
    PATCH=25,                             # samples per token (0.25 s at 100 Hz)
    N_FOLDS=5, FOLDS=[0], SEED=42,
    MODELS=["mamba", "transformer", "gru", "cnn"],
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    EPOCHS=15, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    TRAIN_MIN_VOTES=0,
    HIGH_VOTE=10,                         # threshold for the high-agreement evaluation subset
    NUM_WORKERS=4, AMP=True,
    USE_OFFICIAL_MAMBA2=False,
    RUN_BENCHMARK=True,
    BENCH_CTX=[50, 100, 200, 400, 600, 1200],
)

In [2]:
import pyarrow.parquet as pq
import multiprocessing as mp
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
MP = mp.get_context("fork")   # Kaggle is Linux; fork shares globals with workers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")

ROOT = find_root()
print("data:", ROOT)

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Labels and patient-level folds

In [3]:
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
P_COLS = [f"p_{c}" for c in CLASSES]
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0   # experts labelled the central 10 s of each 50 s window

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k
assert df.groupby("patient_id")["fold"].nunique().max() == 1, "patient leaked across folds"

print(f"rows={len(df):,}  eegs={df.eeg_id.nunique():,}  patients={df.patient_id.nunique():,}")
print(df.groupby("fold").agg(rows=("eeg_id", "size"), eegs=("eeg_id", "nunique"), patients=("patient_id", "nunique")))
print(df.drop_duplicates("eeg_id")["expert_consensus"].value_counts())

rows=106,800  eegs=17,089  patients=1,950
       rows  eegs  patients
fold                       
0     17085  2795       383
1     25502  3689       386
2     21174  3585       394
3     22141  3238       397
4     20898  3782       390
expert_consensus
Other      7196
Seizure    2785
LPD        2577
GRDA       1828
GPD        1787
LRDA        916
Name: count, dtype: int64


## Preprocessing
18-channel bipolar double banana + midline (identical to I-CARE's 18 bipolar pairs). Band-pass 0.5–40 Hz at 200 Hz, then every second sample (100 Hz; the 40 Hz low-pass prevents aliasing). Amplitudes stay in µV (no per-window z-scoring), because absolute voltage — suppression, high-amplitude discharges — is clinically informative.

In [4]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),    # left temporal chain
           ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),    # right temporal chain
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),    # left parasagittal chain
           ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),    # right parasagittal chain
           ("Fz", "Cz"), ("Cz", "Pz")]                                 # midline
FLIP_PERM = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11, 16, 17]   # left <-> right hemisphere
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH = len(BIPOLAR)
SRC_FS = 200
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")
CLIP_UV = 500.0

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def _nrows(eid):
    return pq.ParquetFile(f"{ROOT}/train_eegs/{eid}.parquet").metadata.num_rows

def plan_cache(df, half_sec):
    g = df.groupby("eeg_id")["center"].agg(["min", "max"]).reset_index()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        n = pool.map(_nrows, g["eeg_id"].tolist(), chunksize=64)
    g["n_src"] = n
    s0 = np.floor((g["min"] - half_sec) * SRC_FS).clip(lower=0).astype(np.int64)
    s0 -= s0 % 2
    s1 = np.minimum(g["n_src"], np.ceil((g["max"] + half_sec) * SRC_FS)).astype(np.int64)
    g["s0"], g["length"] = s0, ((s1 - s0 + 1) // 2).clip(lower=0)
    g["offset"] = np.concatenate([[0], np.cumsum(g["length"].to_numpy())[:-1]])
    return g[["eeg_id", "n_src", "s0", "length", "offset"]]

def _build_one(args):
    eid, s0, length = args
    x = load_bipolar(f"{ROOT}/train_eegs/{eid}.parquet")[s0::2][:length]
    out = np.zeros((length, N_CH), np.float16)
    out[:len(x)] = x
    return out

def build_cache(df):
    meta_path = f"{CFG['CACHE_DIR']}/meta.json"
    want = dict(cache_ctx=CFG["CACHE_CTX_SEC"], jitter=CFG["JITTER_SEC"], fs=CFG["FS"], n_eegs=int(df.eeg_id.nunique()))
    if os.path.exists(meta_path) and json.load(open(meta_path)) == want:
        print("cache found, reusing")
        return pd.read_csv(f"{CFG['CACHE_DIR']}/index.csv")
    half = CFG["CACHE_CTX_SEC"] / 2 + CFG["JITTER_SEC"]
    idx = plan_cache(df, half)
    total = int(idx["length"].sum())
    need = total * N_CH * 2
    free = shutil.disk_usage(CFG["CACHE_DIR"]).free
    print(f"cache: {total/CFG['FS']/3600:,.1f} h of EEG, {need/1e9:.2f} GB needed, {free/1e9:.1f} GB free")
    if need > 0.9 * free:
        raise RuntimeError("Not enough disk: lower CACHE_CTX_SEC or point CACHE_DIR at a larger disk")
    mm = np.lib.format.open_memmap(f"{CFG['CACHE_DIR']}/signals.npy", mode="w+", dtype=np.float16, shape=(total, N_CH))
    jobs = list(zip(idx["eeg_id"], idx["s0"], idx["length"]))
    t0 = time.time()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        for i, (x, off) in enumerate(zip(pool.imap(_build_one, jobs, chunksize=16), idx["offset"])):
            mm[off:off + len(x)] = x
            if i % 2000 == 0:
                print(f"  {i:,}/{len(jobs):,}  {time.time()-t0:.0f}s")
    mm.flush(); del mm
    idx.to_csv(f"{CFG['CACHE_DIR']}/index.csv", index=False)
    json.dump(want, open(meta_path, "w"))
    print(f"cache built in {time.time()-t0:.0f}s")
    return idx

## Dataset
Each item is a `CTX_SEC` window centred on the labelled 10 s. Positions outside the recording are zero-filled and flagged in a mask; the SSM treats masked steps as identity transitions (Δ = 0), the Transformer ignores them via key-padding, the readout never pools them.
Training samples one random sub-window per EEG per epoch, so long recordings with many overlapping sub-windows don't dominate. Validation uses the middle sub-window of each EEG (the competition test set also has one row per EEG).

In [5]:
class HMSDataset(Dataset):
    """Self-contained (no notebook globals in __getitem__), so it works with any worker start method."""
    def __init__(self, rows, ctx_sec, train):
        assert ctx_sec <= CFG["CACHE_CTX_SEC"], "CTX_SEC exceeds the cached context; rebuild the cache"
        self.eid = rows["eeg_id"].to_numpy()
        self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.y = rows["y"].to_numpy()
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs)
        assert self.n % CFG["PATCH"] == 0
        self.train, self.jitter, self.flip = train, CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"
        self.mm = None

    def __len__(self):
        return len(self.eid)

    def __getitem__(self, i):
        if self.mm is None:                      # open lazily so each worker gets its own handle
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i]
        if self.train:
            c += np.random.uniform(-self.jitter, self.jitter)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        x = np.zeros((self.n, self.n_ch), np.float32)
        m = np.zeros(self.n, np.float32)
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b > a:
            x[a - i0:b - i0] = self.mm[off + a:off + b]
            m[a - i0:b - i0] = 1.0
        x = x.T / 50.0                           # ~unit scale while keeping absolute amplitude
        if self.train:
            if np.random.rand() < 0.5:
                x = x[self.flip]
            x = x * np.random.uniform(0.9, 1.1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(m), torch.from_numpy(self.p[i]), int(self.y[i])

class OnePerEEG(Sampler):
    def __init__(self, eids, seed):
        order = np.argsort(eids, kind="stable")
        _, starts = np.unique(eids[order], return_index=True)
        self.groups = np.split(order, starts[1:])
        self.seed, self.epoch = seed, 0
    def __len__(self):
        return len(self.groups)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = np.array([g[rng.integers(len(g))] for g in self.groups])
        rng.shuffle(idx)
        return iter(idx.tolist())

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]

## Models
All models: `Stem` (conv + 0.25 s patching → tokens) → sequence mixer → readout (pool over the central 10 s ⊕ pool over the full context) → 6 logits. Only the mixer differs.

The SSM is a bidirectional Mamba-2 block. `ssd()` is the chunked state-space-duality algorithm (exact); `ssd_reference()` is the plain recurrence used to verify it. `Mamba2.step()` is the recurrent form used for streaming.

In [6]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## v6 settings

In [7]:
CFG.update(dict(
    CACHE_CTX_SEC=200, CACHE_DIR="/tmp/hms_cache", RESULTS_DIR="/kaggle/working/results_v6",
    PREV_GLOBS=["/kaggle/input/**/results_v6/*.json", "/kaggle/input/**/results_v6/*_valpred.npy"],   # resume across sessions
    V5_GLOBS=["/kaggle/input/**/results_v5/*_valpred.npy"],          # seed-42 BA predictions from v5
    CTX=200, N_FOLDS=5, EPOCHS=6, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0, TOKENS_PER_BATCH=25600,
    DROPOUT=0.1, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False,
    P_TRUNC=0.5, W_CLIP=(0.25, 4.0),
    # (model, variant, training seed), in priority order. Seed 42 = the seed of all earlier runs; folds never change.
    RUNS=[("gru", "ba_noreweight", 42), ("transformer", "ba_noreweight", 42),
          ("mamba", "ba_noreweight", 1042), ("gru", "ba_noreweight", 1042), ("transformer", "ba_noreweight", 1042),
          ("mamba", "ba_noreweight", 2042), ("gru", "ba_noreweight", 2042), ("transformer", "ba_noreweight", 2042)],
    EST_MIN_PER_FOLD=dict(mamba=16.5, gru=4.2, transformer=5.2),
    TIME_BUDGET_H=4.5,
    B=4000, MIN_N=20, T_GRID=list(np.round(np.linspace(0.5, 3.0, 51), 3)),
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
))
CFG_OVERRIDES = {"TIME_BUDGET_H": 2.5, "RUNS": [
   ("gru", "ba_noreweight", 42), ("transformer", "ba_noreweight", 42),
   ("mamba", "ba_noreweight", 1042), ("gru", "ba_noreweight", 1042), ("transformer", "ba_noreweight", 1042),
   ("mamba", "ba_noreweight", 2042), ("gru", "ba_noreweight", 2042), ("transformer", "ba_noreweight", 2042),
   ("mamba", "ba_noreweight", 3042), ("gru", "ba_noreweight", 3042)]}
CFG.update(CFG_OVERRIDES)
T_START = time.time()
os.makedirs(CFG["CACHE_DIR"], exist_ok=True); os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)

n_copied = 0
for pat in CFG["PREV_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        dst = os.path.join(CFG["RESULTS_DIR"], os.path.basename(fp))
        if not os.path.exists(dst):
            shutil.copy(fp, dst); n_copied += 1
print(f"resumed {n_copied} files from earlier v6 outputs")
V5 = {}
for pat in CFG["V5_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        V5.setdefault(os.path.basename(fp), fp)
print(f"found {len(V5)} v5 prediction files")
PREDS = {}

def run_tag(name, variant, seed, fold):
    return f"{name}_{variant}_ctx{CFG['CTX']}_f{fold}" + ("" if seed == 42 else f"_seed{seed}")

resumed 120 files from earlier v6 outputs
found 40 v5 prediction files


## Per-EEG and per-row context availability

In [8]:
cache_idx = build_cache(df)
CACHE = {int(r.eeg_id): (int(r.offset), int(r.s0), int(r.length)) for r in cache_idx.itertuples()}
DUR = dict(zip(cache_idx["eeg_id"].astype(int), cache_idx["n_src"] / SRC_FS))

def extra_context(center, dur, ctx=CFG["CTX"]):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(dur - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(dur - center, 25), 0, None)

df["dur_s"] = df["eeg_id"].map(DUR)
df["extra_s"] = extra_context(df["center"].to_numpy(), df["dur_s"].to_numpy())
df["ctx_bin"] = pd.cut(df["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)

VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)
T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); PID = OOF["patient_id"].to_numpy(); FOLD = OOF["fold"].to_numpy()
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(NV >= lo) & (NV <= hi), "vote_group"] = name
OOF["label"] = np.array(CLASSES)[OOF["y"].to_numpy()]
VG, CB = OOF["vote_group"].to_numpy(), OOF["ctx_bin"].to_numpy()
print(pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).to_string())

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)

def load_oof(model, ctx, stage):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(f"{model}_ctx{ctx}_f{f}_{stage}_valpred.npy")
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

UP, INV = np.unique(PID, return_inverse=True)
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)

def boot(d, msk):
    if msk.sum() == 0:
        return (float("nan"),) * 4
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (W @ s_) / (W @ n_)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return float(d[msk].mean()), float("nan"), float("nan"), float("nan")
    lo, hi = np.percentile(r, [2.5, 97.5])
    return float(d[msk].mean()), lo, hi, max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    idx = np.argsort(np.where(ok, p, np.inf)); k = ok.sum(); run = 0.0
    for i_, j in enumerate(idx[:k]):
        run = max(run, min(1.0, (k - i_) * p[j])); out[j] = run
    return out

SUBSETS = {"all EEGs": np.ones(len(OOF), bool)}
for b in CFG["CTX_LABELS"]:
    SUBSETS[f"all | {b}"] = CB == b
for g in ["1-2 votes", "3-9 votes", ">=10 votes"]:
    SUBSETS[f"{g} | all"] = VG == g
    for b in CFG["CTX_LABELS"]:
        SUBSETS[f"{g} | {b}"] = (VG == g) & (CB == b)

cache: 380.3 h of EEG, 4.93 GB needed, 1038.2 GB free
  0/17,089  1s
  2,000/17,089  20s
  4,000/17,089  41s
  6,000/17,089  61s
  8,000/17,089  81s
  10,000/17,089  101s
  12,000/17,089  120s
  14,000/17,089  141s
  16,000/17,089  162s
cache built in 174s
ctx_bin     full (>=140 s)  none (<10 s)  partial (10-140 s)    All
vote_group                                                         
1-2 votes               25          4526                 296   4847
3-9 votes              297          2298                4047   6642
>=10 votes             169          2648                2783   5600
All                    491          9472                7126  17089


## B. Boundary-agnostic (BA) training of 200 s models
Three changes make the recording boundary invisible or useless to the model:
1. **Reflect-fill, no mask:** the part of the 200 s window outside the recording is filled by mirroring the real signal, and the mask says "real" everywhere — so padding can no longer reveal where the recording ends.
2. **Random truncation (p = 0.5):** during training, part of the real context on each side is hidden and mirror-filled too, so mirrored regions also appear in long recordings and stop being a "short recording" signature. The central 50 s is never hidden.
3. **Label balancing across context availability (variant `ba`):** each training row is weighted by Σ_k t_k · π_k / π_{k | context bin}, removing the incentive to infer the label from how much context exists. `ba_noreweight` drops this step (ablation).

Evaluation uses reflect-fill without truncation (`ba`) and, for the residual-shortcut test, the central 50 s only, mirror-filled (`trunc50`).

In [9]:
class BADataset(Dataset):
    def __init__(self, rows, ctx_sec, train, mode="ba", weights=None):
        self.eid = rows["eeg_id"].to_numpy(); self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32); self.y = rows["y"].to_numpy()
        self.w = np.ones(len(rows), np.float32) if weights is None else np.asarray(weights, np.float32)
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs); self.train, self.mode = train, mode
        self.jitter, self.p_trunc, self.flip = CFG["JITTER_SEC"], CFG["P_TRUNC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"; self.mm = None
    def __len__(self):
        return len(self.eid)
    def window(self, i, rng):
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i] + (rng.uniform(-self.jitter, self.jitter) if self.train else 0.0)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        a, b = max(i0, 0), min(i0 + self.n, length)
        lo, hi = a - i0, b - i0                               # real signal inside the window: [lo, hi)
        cidx, h50 = self.n // 2, 25 * self.fs
        if self.mode == "trunc50":
            lo, hi = max(lo, cidx - h50), min(hi, cidx + h50)
        elif self.train and rng.random() < self.p_trunc:
            lo = int(rng.integers(lo, max(lo, cidx - h50) + 1))
            hi = int(rng.integers(min(hi, cidx + h50), hi + 1))
        if hi <= lo:
            return np.zeros((self.n, self.n_ch), np.float32), lo, hi
        real = np.asarray(self.mm[off + i0 + lo:off + i0 + hi], dtype=np.float32)
        x = np.pad(real, ((lo, self.n - hi), (0, 0)), mode="symmetric") if (lo > 0 or hi < self.n) else real
        return x, lo, hi
    def __getitem__(self, i):
        if self.mm is None:
            self.mm = np.load(self.path, mmap_mode="r")
        rng = np.random.default_rng()
        x, lo, hi = self.window(i, rng)
        m = np.ones(self.n, np.float32) if hi > lo else np.zeros(self.n, np.float32)
        x = x.T / 50.0
        if self.train:
            if rng.random() < 0.5:
                x = x[self.flip]
            x = x * rng.uniform(0.9, 1.1)
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), torch.from_numpy(m),
                torch.from_numpy(self.p[i]), torch.tensor(self.w[i]))

# sanity: the central 50 s is always real signal, the mask never reveals the boundary
_ds = BADataset(OOF.head(200), CFG["CTX"], train=True)
_ds.mm = np.load(_ds.path, mmap_mode="r")
_rng = np.random.default_rng(0)
for i in range(min(200, len(_ds))):
    _, lo, hi = _ds.window(i, _rng)
    tol = int(CFG["JITTER_SEC"] * CFG["FS"]) + 1                      # jitter can push the label edge by up to 1 s
    assert lo <= _ds.n // 2 - 25 * CFG["FS"] + tol and hi >= _ds.n // 2 + 25 * CFG["FS"] - tol, (i, lo, hi)
print("BA sanity check passed: central 50 s always real; mask uniform")

def balance_weights(rows):
    """w = sum_k t_k * pi_k / pi_{k|bin}: removes label information carried by context availability."""
    Tr = rows[P_COLS].to_numpy(np.float64)
    pi = Tr.mean(0)
    w = np.zeros(len(rows))
    for b in rows["ctx_bin"].unique():
        msk = (rows["ctx_bin"] == b).to_numpy()
        pib = np.clip(Tr[msk].mean(0), 1e-4, None)
        w[msk] = (Tr[msk] * (pi / pib)).sum(1)
    w = np.clip(w, *CFG["W_CLIP"])
    return w / w.mean()

def param_groups(model):
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim < 2 or n_.endswith(("A_log", "D", "dt_bias")) else decay).append(p)
    return [dict(params=decay, weight_decay=CFG["WD"]), dict(params=no_decay, weight_decay=0.0)]

def loader(ds, batch, sampler=None, drop_last=False):
    return DataLoader(ds, batch, sampler=sampler, shuffle=False, drop_last=drop_last, num_workers=CFG["NUM_WORKERS"],
                      pin_memory=DEV.type == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0,
                      multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, m, _, _ in dl:
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
            out.append(torch.softmax(model(x.to(DEV), m.to(DEV)).float(), -1).cpu())
    return torch.cat(out).numpy()

def train_ba(name, variant, seed, fold):
    tag = run_tag(name, variant, seed, fold)
    seed_all(seed + fold)
    tr = df[df.fold != fold].reset_index(drop=True)
    w = balance_weights(tr) if variant == "ba" else None
    tok = CFG["CTX"] * CFG["FS"] // CFG["PATCH"]
    micro = max(1, min(CFG["BATCH"], CFG["TOKENS_PER_BATCH"] // tok)); accum = math.ceil(CFG["BATCH"] / micro)
    dl = loader(BADataset(tr, CFG["CTX"], train=True, weights=w), micro, OnePerEEG(tr["eeg_id"].to_numpy(), seed + fold), drop_last=True)
    model = Net(name).to(DEV)
    opt = torch.optim.AdamW(param_groups(model), lr=CFG["LR"])
    total = max(1, CFG["EPOCHS"] * (len(dl) // accum)); warm = max(1, int(CFG["WARMUP_FRAC"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    use_amp = CFG["AMP"] and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"\n=== {tag} | seed {seed} | micro {micro} x accum {accum} | weights {'on' if w is not None else 'off'}")
    t_run, hist = time.time(), []
    for ep in range(CFG["EPOCHS"]):
        model.train(); t0 = time.time(); tl = nb = 0.0
        opt.zero_grad(set_to_none=True)
        for i, (x, m, p, wb) in enumerate(dl):
            x, m, p, wb = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True), wb.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, m)
            kl = F.kl_div(F.log_softmax(logits.float(), -1), p, reduction="none").sum(1)
            loss = (kl * wb).mean()
            scaler.scale(loss / accum).backward()
            tl += loss.item(); nb += 1
            if (i + 1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), sec=time.time() - t0))
        print(f"  {tag} ep {ep} | train KL {tl/max(nb,1):.4f} | {time.time()-t0:.0f}s")
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"{CFG['RESULTS_DIR']}/{tag}.pt")
    for mode in ("ba", "trunc50"):
        P = predict(model, loader(BADataset(VA[fold], CFG["CTX"], train=False, mode=mode), micro * 2))
        np.save(f"{CFG['RESULTS_DIR']}/{tag}_{mode}_valpred.npy", P)
    json.dump(dict(model=name, variant=variant, seed=seed, fold=fold, ctx=CFG["CTX"], minutes=(time.time() - t_run) / 60, history=hist),
              open(f"{CFG['RESULTS_DIR']}/{tag}.json", "w"), indent=1)
    print(f"=== {tag} done in {(time.time()-t_run)/60:.1f} min")
    del model
    if DEV.type == "cuda": torch.cuda.empty_cache()

def done(name, variant, seed, fold):
    tag = run_tag(name, variant, seed, fold)
    return all(os.path.exists(f"{CFG['RESULTS_DIR']}/{tag}_{m}_valpred.npy") or f"{tag}_{m}_valpred.npy" in V5 for m in ("ba", "trunc50"))

PLAN = [(n_, v_, s_, f_) for n_, v_, s_ in CFG["RUNS"] for f_ in range(CFG["N_FOLDS"])]
todo = [r for r in PLAN if not done(*r)]
print(f"{len(PLAN)} runs planned, {len(PLAN) - len(todo)} already done, "
      f"~{sum(CFG['EST_MIN_PER_FOLD'].get(r[0], 16) for r in todo) / 60:.1f} GPU-h left")
for n_, v_, s_, f_ in todo:
    used = (time.time() - T_START) / 3600
    est = CFG["EST_MIN_PER_FOLD"].get(n_, 16) / 60
    if used + est > CFG["TIME_BUDGET_H"]:
        print(f"time budget: stopping before {run_tag(n_, v_, s_, f_)} ({used:.2f} h used). Attach this output and rerun to continue.")
        break
    train_ba(n_, v_, s_, f_)

BA sanity check passed: central 50 s always real; mask uniform
50 runs planned, 40 already done, ~1.7 GPU-h left

=== mamba_ba_noreweight_ctx200_f0_seed3042 | seed 3042 | micro 32 x accum 2 | weights off
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 0 | train KL 0.9341 | 178s
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 1 | train KL 0.6762 | 176s
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 2 | train KL 0.5960 | 178s
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 3 | train KL 0.5355 | 177s
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 4 | train KL 0.4827 | 177s
  mamba_ba_noreweight_ctx200_f0_seed3042 ep 5 | train KL 0.4560 | 177s
=== mamba_ba_noreweight_ctx200_f0_seed3042 done in 18.2 min

=== mamba_ba_noreweight_ctx200_f1_seed3042 | seed 3042 | micro 32 x accum 2 | weights off
  mamba_ba_noreweight_ctx200_f1_seed3042 ep 0 | train KL 0.9356 | 165s
  mamba_ba_noreweight_ctx200_f1_seed3042 ep 1 | train KL 0.6915 | 164s
  mamba_ba_noreweight_ctx200_f1_seed3042 ep 2 | train KL 0.6119 | 165s
  mamb

## Results across seeds
Pooled out-of-fold, calibrated (cross-fitted temperature), patient-level cluster bootstrap. A (model, seed) set is used only when all 5 folds exist.

In [10]:
def load_set(name, variant, seed, mode="ba"):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        b = f"{run_tag(name, variant, seed, f)}_{mode}_valpred.npy"
        fp = f"{CFG['RESULTS_DIR']}/{b}" if os.path.exists(f"{CFG['RESULTS_DIR']}/{b}") else V5.get(b)
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def temper(P, t):
    L = np.log(np.clip(P, 1e-12, 1)) / t; L -= L.max(1, keepdims=True); E = np.exp(L)
    return E / E.sum(1, keepdims=True)

def calibrate(P):
    out = np.empty_like(P)
    for f in range(CFG["N_FOLDS"]):
        tr, te = FOLD != f, FOLD == f
        best = min(CFG["T_GRID"], key=lambda t: (T[tr] * (np.log(T[tr] + 1e-15) - np.log(temper(P[tr], t) + 1e-15))).sum(1).mean())
        out[te] = temper(P[te], best)
    return out

MODELS_ = ["mamba", "gru", "transformer"]
SEEDS_ = sorted({s_ for _, _, s_ in CFG["RUNS"]} | {42})
KL, RAW = {}, {}
for name in MODELS_:
    for variant in ("ba", "ba_noreweight"):
        for seed in SEEDS_:
            P = load_set(name, variant, seed)
            if P is not None:
                RAW[(name, variant, seed)] = per_eeg_kl(P)
                KL[(name, variant, seed)] = per_eeg_kl(calibrate(P))
print("available sets:", sorted(KL))

per_seed = pd.DataFrame([dict(model=n_, variant=v_, seed=s_, kl_raw=RAW[(n_, v_, s_)].mean(), kl_cal=k.mean(),
                              kl_cal_ge10=k[NV >= 10].mean()) for (n_, v_, s_), k in KL.items()])
per_seed.to_csv(f"{CFG['RESULTS_DIR']}/v6_per_seed.csv", index=False)
print("### Pooled OOF KL per training seed\n" + per_seed.round(4).to_string(index=False))
if len(per_seed):
    agg = per_seed.groupby(["model", "variant"]).agg(n_seeds=("seed", "size"), kl_cal_mean=("kl_cal", "mean"), kl_cal_sd=("kl_cal", "std"),
                                                    ge10_mean=("kl_cal_ge10", "mean"), ge10_sd=("kl_cal_ge10", "std")).reset_index()
    agg.to_csv(f"{CFG['RESULTS_DIR']}/v6_seed_summary.csv", index=False)
    print("\n### Mean and SD across seeds (calibrated)\n" + agg.round(4).to_string(index=False))

available sets: [('gru', 'ba', 42), ('gru', 'ba_noreweight', 42), ('gru', 'ba_noreweight', 1042), ('gru', 'ba_noreweight', 2042), ('gru', 'ba_noreweight', 3042), ('mamba', 'ba', 42), ('mamba', 'ba_noreweight', 42), ('mamba', 'ba_noreweight', 1042), ('mamba', 'ba_noreweight', 2042), ('mamba', 'ba_noreweight', 3042), ('transformer', 'ba', 42), ('transformer', 'ba_noreweight', 42), ('transformer', 'ba_noreweight', 1042), ('transformer', 'ba_noreweight', 2042)]
### Pooled OOF KL per training seed
      model       variant  seed  kl_raw  kl_cal  kl_cal_ge10
      mamba            ba    42  0.6967  0.6893       0.5479
      mamba ba_noreweight    42  0.6889  0.6727       0.5157
      mamba ba_noreweight  1042  0.6673  0.6552       0.5006
      mamba ba_noreweight  2042  0.6900  0.6750       0.5144
      mamba ba_noreweight  3042  0.6768  0.6625       0.5036
        gru            ba    42  0.7195  0.7184       0.5815
        gru ba_noreweight    42  0.6877  0.6832       0.5285
        gru ba

### Architecture comparison with the same recipe (no reweighting)
Per seed, and on the per-EEG KL averaged over the seeds both models share (the main test; Holm-corrected).

In [11]:
rows = []
V_ = "ba_noreweight"
for a_, b_ in [("mamba", "gru"), ("mamba", "transformer"), ("gru", "transformer")]:
    common = [s_ for s_ in SEEDS_ if (a_, V_, s_) in KL and (b_, V_, s_) in KL]
    tests = [(f"seed {s_}", KL[(a_, V_, s_)], KL[(b_, V_, s_)]) for s_ in common]
    if len(common) > 1:
        tests.append((f"mean of {len(common)} seeds", np.mean([KL[(a_, V_, s_)] for s_ in common], 0), np.mean([KL[(b_, V_, s_)] for s_ in common], 0)))
    for lab, ka, kb in tests:
        for sname, msk in (("all EEGs", np.ones(len(OOF), bool)), (">=10 votes", NV >= 10)):
            m_, lo, hi, p = boot(ka - kb, msk)
            rows.append(dict(comparison=f"{a_} - {b_}", seeds=lab, subset=sname, n=int(msk.sum()), kl_a=ka[msk].mean(), kl_b=kb[msk].mean(),
                             diff=m_, ci_lo=lo, ci_hi=hi, rel_pct=100 * m_ / kb[msk].mean(), p=p))
arch = pd.DataFrame(rows)
if len(arch):
    main = arch["seeds"].str.startswith("mean")
    if not main.any():
        main = arch["seeds"] == "seed 42"
    arch["p_holm"] = np.nan
    arch.loc[main, "p_holm"] = holm(arch.loc[main, "p"].to_numpy())
arch.to_csv(f"{CFG['RESULTS_DIR']}/v6_architecture.csv", index=False)
print("### Calibrated KL differences, same recipe (negative = first better)\n" + arch.round(4).to_string(index=False))

### Calibrated KL differences, same recipe (negative = first better)
         comparison           seeds     subset     n   kl_a   kl_b    diff   ci_lo   ci_hi  rel_pct      p  p_holm
        mamba - gru         seed 42   all EEGs 17089 0.6727 0.6832 -0.0105 -0.0242  0.0037  -1.5324 0.1460     NaN
        mamba - gru         seed 42 >=10 votes  5600 0.5157 0.5285 -0.0128 -0.0306  0.0046  -2.4254 0.1570     NaN
        mamba - gru       seed 1042   all EEGs 17089 0.6552 0.6816 -0.0264 -0.0431 -0.0092  -3.8674 0.0020     NaN
        mamba - gru       seed 1042 >=10 votes  5600 0.5006 0.5228 -0.0222 -0.0392 -0.0045  -4.2426 0.0165     NaN
        mamba - gru       seed 2042   all EEGs 17089 0.6750 0.6858 -0.0108 -0.0353  0.0200  -1.5745 0.4150     NaN
        mamba - gru       seed 2042 >=10 votes  5600 0.5144 0.5266 -0.0122 -0.0298  0.0078  -2.3183 0.2135     NaN
        mamba - gru       seed 3042   all EEGs 17089 0.6625 0.6903 -0.0278 -0.0519 -0.0003  -4.0266 0.0480     NaN
        mam

### Reweighting ablation for every model (seed 42) and the shortcut check for the new runs

In [12]:
rows = []
for name in MODELS_:
    if (name, "ba_noreweight", 42) in KL and (name, "ba", 42) in KL:
        for sname, msk in (("all EEGs", np.ones(len(OOF), bool)), (">=10 votes", NV >= 10)):
            m_, lo, hi, p = boot(KL[(name, "ba_noreweight", 42)] - KL[(name, "ba", 42)], msk)
            rows.append(dict(model=name, subset=sname, kl_reweight=KL[(name, "ba", 42)][msk].mean(), kl_noreweight=KL[(name, "ba_noreweight", 42)][msk].mean(),
                             diff=m_, ci_lo=lo, ci_hi=hi, rel_pct=100 * m_ / KL[(name, "ba", 42)][msk].mean(), p=p))
rw = pd.DataFrame(rows); rw.to_csv(f"{CFG['RESULTS_DIR']}/v6_reweight_ablation.csv", index=False)
print("### no-reweight minus reweight (negative = no-reweight better)\n" + rw.round(4).to_string(index=False))

shift = []
for (n_, v_, s_) in KL:
    a_, b_ = load_set(n_, v_, s_, "ba"), load_set(n_, v_, s_, "trunc50")
    if b_ is None:
        continue
    msk = CB == CFG["CTX_LABELS"][2]
    if msk.sum() >= CFG["MIN_N"]:
        shift.append(dict(model=f"{n_} {v_} seed {s_}", n_full=int(msk.sum()), **{f"d_{c}": (b_[msk, k] - a_[msk, k]).mean() for k, c in enumerate(CLASSES)}))
if shift:
    print("\n### Probability shift when truncated to the central 50 s, full-context EEGs (shortcut models: Seizure -0.21, Other +0.24)\n"
          + pd.DataFrame(shift).round(3).to_string(index=False))
print("outputs:", sorted(os.listdir(CFG["RESULTS_DIR"]))[:60])

### no-reweight minus reweight (negative = no-reweight better)
      model     subset  kl_reweight  kl_noreweight    diff   ci_lo   ci_hi  rel_pct      p
      mamba   all EEGs       0.6893         0.6727 -0.0166 -0.0314 -0.0027  -2.4050 0.0185
      mamba >=10 votes       0.5479         0.5157 -0.0323 -0.0465 -0.0186  -5.8866 0.0002
        gru   all EEGs       0.7184         0.6832 -0.0353 -0.0495 -0.0213  -4.9098 0.0002
        gru >=10 votes       0.5815         0.5285 -0.0531 -0.0690 -0.0372  -9.1235 0.0002
transformer   all EEGs       0.7584         0.7324 -0.0260 -0.0382 -0.0140  -3.4240 0.0002
transformer >=10 votes       0.6157         0.5785 -0.0372 -0.0533 -0.0215  -6.0358 0.0002

### Probability shift when truncated to the central 50 s, full-context EEGs (shortcut models: Seizure -0.21, Other +0.24)
                              model  n_full  d_Seizure  d_LPD  d_GPD  d_LRDA  d_GRDA  d_Other
                   mamba ba seed 42     491     -0.015  0.008  0.002  -0.001  -0.00